# Your First LLM Call
*ICS 603 — lecture 1.3 'Course Arc Preview and First LLM Taste'*

One tiny call to a large language model — the same `ask_model()` function the food-safety assistant uses in production. No retrieval, no grounding yet: just prompt in, text out.

In [1]:
# Load the endpoint settings from ../.env into the environment (key never printed).
import os
from pathlib import Path

for line in Path('../.env').read_text(encoding='utf-8').splitlines():
    if '=' in line and not line.lstrip().startswith('#'):
        key, _, value = line.partition('=')
        os.environ.setdefault(key.strip(), value.strip())

print('endpoint:', os.environ.get('LLM_BASE_URL', '(default)'))
print('model:', os.environ.get('LLM_MODEL', '(default)'))
print('key loaded:', 'LLM_API_KEY' in os.environ)  # True/False only - never the key

endpoint: https://llm.jetstream-cloud.org/api
model: gpt-oss-120b
key loaded: True


In [2]:
# THE LIVE CALL - one prompt, one response. Also saves the response so the
# fallback cell below works offline next time.
from pathlib import Path
from foodsafety_rag.generate import ask_model

response = ask_model('In one sentence, what is retrieval-augmented generation?')
Path('../fixtures/first_llm_call.txt').write_text(response, encoding='utf-8')
print(response)

Retrieval‑augmented generation is a language‑model approach that first fetches relevant external documents or facts and then generates its response conditioned on both the retrieved information and the original prompt.


## What the response object carries
`ask_model` hands back a string, but the endpoint returns more than that: how many tokens the call used and why the model stopped. Before running the cell, predict the prompt-token count for a question of about ten words.

In [3]:
# The same call, one level down - the response object instead of the text.
import time
from foodsafety_rag.config import get_settings
from foodsafety_rag.generate import get_client

client, model = get_client(), get_settings().llm_model
prompt = 'In one sentence, what is retrieval-augmented generation?'

start = time.monotonic()
raw = client.chat.completions.create(
    model=model, messages=[{'role': 'user', 'content': prompt}])
elapsed = time.monotonic() - start

print('prompt tokens:    ', raw.usage.prompt_tokens)
print('completion tokens:', raw.usage.completion_tokens)
print('latency:           %.2f s' % elapsed)
print('finish reason:    ', raw.choices[0].finish_reason)

prompt tokens:     81
completion tokens: 83
latency:           0.91 s
finish reason:     stop


In [4]:
# How much of that context did we write? Send one word and compare.
brief = client.chat.completions.create(
    model=model, messages=[{'role': 'user', 'content': 'Hi'}], max_tokens=1)
print('prompt tokens for the single word "Hi":', brief.usage.prompt_tokens)
# The difference from zero is the chat template the server adds for us.

prompt tokens for the single word "Hi": 70


## Fallback: captured response (no key / no network)
If the live call above fails in class (no key, no wifi, quota), the cell below replays the response captured on a previous successful run. Same text, zero network.

In [5]:
from pathlib import Path

captured = Path('../fixtures/first_llm_call.txt').read_text(encoding='utf-8')
print('[captured response - replayed offline]')
print(captured)

[captured response - replayed offline]
Retrieval‑augmented generation is a language‑model approach that first fetches relevant external documents or facts and then generates its response conditioned on both the retrieved information and the original prompt.


## What to notice

| | This call |
|---|---|
| **Input** | One sentence of plain English - no code, no schema. |
| **Context** | Not zero. The server wraps our message in a chat template before the model sees it - measure it with the cells above. RAG is how we choose the rest. |
| **Output** | Fluent prose. Fluent is not the same as verified. |
| **Cost** | Metered per token, unlike an ordinary function call - and the prompt is charged too, not just the answer. |
| **Latency** | ~1-3 seconds - orders of magnitude slower than a normal function call; design around it. |
| **Uncertainty** | Re-run the live cell: the wording can change. Same input, different output - ordinary software never does this. |

The rest of the course wraps this one call in retrieval (M8), structure (Pydantic, M4/M6), storage (M7), and deployment (Docker/Jetstream) - turning 'useful' into 'trustworthy'.